# Notebook 09 — MoE-Tiny: Mixture of Experts at VSLM scale

*Sparse routing through 4 specialist experts per layer. Candidate for the 'generative' slot in the dual-mode 6502 build.*

## What we're doing and why

In nb07b, the entire MLP processes every token the same way. **Every token pays for the full MLP** regardless of whether it needs that capacity.

MoE replaces the dense MLP with **N specialized experts**. A learned router picks which expert(s) to use per token. At inference (top-1), **each token only activates 1 expert** — so you get the *capacity of N experts* for the *compute of 1*.

This is the architecture behind Switch Transformer, Mixtral, GShard, GLaM. Standard at large scale; nearly unexplored at our scale (~30 K params total). We're inventing the recipe.

### Why MoE could help at VSLM

The fundamental constraint: 32 KB EEPROM = ~30 K params. If you spend it all on one dense MLP, the MLP has to be a generalist — it averages over every distinct phenomenon in Shakespeare (dialogue, action, archaic forms, punctuation, etc.).

MoE lets the same parameter budget host **specialists**: one expert that handles dialogue, another for verbs, another for punctuation. Each is smaller individually but, in aggregate, covers more linguistic phenomena.

At small scale this is **theoretical**. We'll empirically check whether it actually beats nb07b's dense transformer.

### The two new concepts you'll learn

1. **Sparse routing** — picking 1 of N experts per token. Discrete decision in the middle of a neural network. Trained via a router network whose softmax becomes the picking distribution.

2. **Load balancing loss** — without it, the router collapses to always picking expert 0 (the #1 MoE failure mode at any scale, especially small). We add an auxiliary loss term that pressures the router toward uniform expert usage across the batch.

### What we'll ship

- `export/wozformer_moe.bin` — the deployable artifact, if MoE beats transformer.
- A direct comparison plot: nb07b val loss vs nb09 val loss, at the same EEPROM budget.
- Expert specialization visualization — which expert handles which tokens.


## Cell 1 — Setup + tokenizer (re-train BPE, same as nb07b/nb08)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, struct
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path

torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

text = Path('../data/tinyshakespeare.txt').read_text().lower()

# BPE training (same algorithm as nb07b)
def train_bpe(text, num_merges):
    EOW = '</w>'
    word_freq = Counter(tuple(list(w) + [EOW]) for w in text.split())
    word_lists = {w: list(w) for w in word_freq}
    merges = []
    for _ in range(num_merges):
        pair_counts = Counter()
        for w, freq in word_freq.items():
            symbols = word_lists[w]
            for i in range(len(symbols)-1):
                pair_counts[(symbols[i], symbols[i+1])] += freq
        if not pair_counts: break
        best, _ = pair_counts.most_common(1)[0]
        new_tok = best[0] + best[1]
        merges.append((best, new_tok))
        for w in word_freq:
            symbols = word_lists[w]
            new_symbols = []
            i = 0
            while i < len(symbols):
                if i < len(symbols)-1 and (symbols[i], symbols[i+1]) == best:
                    new_symbols.append(new_tok); i += 2
                else:
                    new_symbols.append(symbols[i]); i += 1
            word_lists[w] = new_symbols
    vocab_set = set()
    for w in word_freq:
        vocab_set.update(word_lists[w]); vocab_set.update(w)
    return merges, sorted(vocab_set)

EOW = '</w>'
VOCAB_SIZE = 128
merges, vocab = train_bpe(text, 88)
all_toks = ['<unk>'] + sorted(vocab)
while len(all_toks) < VOCAB_SIZE: all_toks.append(f'<pad{len(all_toks)}>')
all_toks = all_toks[:VOCAB_SIZE]
itos = all_toks
stoi = {t: i for i, t in enumerate(itos)}

def encode_word(word):
    symbols = list(word) + [EOW]
    for (a, b), merged in merges:
        i = 0; new_symbols = []
        while i < len(symbols):
            if i < len(symbols)-1 and symbols[i]==a and symbols[i+1]==b:
                new_symbols.append(merged); i += 2
            else:
                new_symbols.append(symbols[i]); i += 1
        symbols = new_symbols
    return [stoi.get(s, 0) for s in symbols]

def encode(text):
    ids = []
    for w in text.split():
        ids.extend(encode_word(w))
    return ids
def decode(ids):
    return ''.join(itos[i] for i in ids).replace(EOW, ' ')

print('encoding corpus...')
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f'tokens: {len(data):,}  (train {len(train_data):,} / val {len(val_data):,})')


## Cell 2 — Hyperparameters

Chosen to fit ~28 KB int8 + headroom for the dual-mode 6502 layout:

- `d_model = 32` (same as nb07b)
- `n_layers = 2` (same as nb07b)
- `n_experts = 4` per layer (new)
- `expert_inner_dim = 16` — bottlenecked. Each expert is `Linear(32 → 16) + ReLU + Linear(16 → 32)`.

### Why bottlenecked experts?

Standard MoE uses experts as wide as the dense MLP they replace. We can't afford that within 32 KB. Bottlenecking forces each expert to learn a *compressed representation* — which actually correlates with specialization (the expert has to throw something away, and the router picks the expert that throws away the *right* something for this token).

### Cycle budget impact

Per-token MLP cost in nb07b: `d_model × inner_dim × 2` = 32 × 64 × 2 = 4,096 mults (with mlp_mult=2 from nb07b's fix).

Per-token MLP cost in MoE-Tiny: `d_model × n_experts` (router) + `d_model × expert_inner × 2` (one expert) = 128 + 32 × 16 × 2 = 1,152 mults.

**MoE per-token MLP is ~3.5× cheaper** in compute despite having 4× the parameters. That's the whole point.


In [ ]:
BATCH_SIZE = 64
BLOCK_SIZE = 32
EMBED_DIM  = 32
NUM_HEADS  = 1
N_LAYERS   = 2
N_EXPERTS  = 4
EXPERT_INNER = 16
LR         = 1e-3
N_STEPS    = 12000
EVAL_EVERY = 500
LB_LOSS_W  = 0.01   # load-balance loss weight

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(0, len(d) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x = torch.stack([d[i:i+BLOCK_SIZE] for i in ix])
    y = torch.stack([d[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x.to(device), y.to(device)


## Cell 3 — The MoE MLP

### The forward pass, two modes

**Training (soft mode):** compute all expert outputs, weighted sum by router softmax. Gradients flow to every expert.

```
router_logits = router(x)                  # (B, T, n_experts)
router_probs  = softmax(router_logits)     # (B, T, n_experts)
expert_outs   = [expert(x) for expert in experts]   # n_experts × (B, T, d_model)
out = sum(router_probs[..., e, None] * expert_outs[e] for e in range(n_experts))
```

**Inference (hard top-1):** pick the single expert with highest router score. Compute only that expert.

```
top1 = router_logits.argmax(dim=-1)        # (B, T)
out = experts[top1](x)                     # but per-token, not per-batch — so we mask
```

In practice on GPU we still compute all experts and mask, because the speedup from skipping unused experts only matters when N is very large. The 6502 firmware, on the other hand, **literally only runs the chosen expert** — that's where the speedup actually pays off.

### The load balance loss

For a batch, let `f_e = fraction of tokens routed to expert e` and `p_e = mean router probability for expert e`. The Switch Transformer auxiliary loss is:

$$\mathcal{L}_{LB} = n_{experts} \cdot \sum_e f_e \cdot p_e$$

Both `f_e` and `p_e` sum to 1 over experts. The product is minimized when both are uniform (= 1/n_experts each). So minimizing this loss pressures the router toward uniform expert usage — preventing collapse.


In [ ]:
class MoEMLP(nn.Module):
    def __init__(self, d_model, n_experts, expert_inner):
        super().__init__()
        self.n_experts = n_experts
        self.router = nn.Linear(d_model, n_experts, bias=False)
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, expert_inner),
                nn.ReLU(),
                nn.Linear(expert_inner, d_model),
            )
            for _ in range(n_experts)
        ])

    def forward(self, x, hard=False):
        # x: (B, T, d_model)
        router_logits = self.router(x)                # (B, T, n_experts)
        router_probs  = F.softmax(router_logits, dim=-1)

        if hard:
            # Top-1 hard routing: pick argmax, run only that expert per token
            top1 = router_probs.argmax(dim=-1)        # (B, T)
            out = torch.zeros_like(x)
            for e in range(self.n_experts):
                mask = (top1 == e).unsqueeze(-1).float()  # (B, T, 1)
                if mask.sum() == 0: continue
                out = out + mask * self.experts[e](x)
        else:
            # Soft mode: weighted mix of all experts
            expert_outs = torch.stack([e(x) for e in self.experts], dim=-2)  # (B, T, n_experts, d_model)
            out = (router_probs.unsqueeze(-1) * expert_outs).sum(dim=-2)

        return out, router_probs

def load_balance_loss(router_probs):
    # router_probs: (B, T, n_experts)
    # f_e = fraction of tokens for which expert e is the top-1 (using argmax for hard counts)
    n = router_probs.shape[-1]
    top1 = router_probs.argmax(dim=-1)
    f = torch.zeros(n, device=router_probs.device)
    for e in range(n):
        f[e] = (top1 == e).float().mean()
    # p_e = mean router probability for expert e (soft)
    p = router_probs.mean(dim=(0, 1))
    return n * (f * p).sum()


## Cell 4 — Attention (unchanged) + the MoE Block

Attention is exactly the same as nb05/nb07b. Only the MLP becomes MoE.


In [ ]:
class Head(nn.Module):
    def __init__(self, embed_dim, head_size, block_size):
        super().__init__()
        self.key   = nn.Linear(embed_dim, head_size, bias=False)
        self.query = nn.Linear(embed_dim, head_size, bias=False)
        self.value = nn.Linear(embed_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.head_size = head_size
    def forward(self, x):
        B, T, _ = x.shape
        k=self.key(x); q=self.query(x); v=self.value(x)
        s = q @ k.transpose(-2, -1) / (self.head_size**0.5)
        s = s.masked_fill(self.tril[:T, :T]==0, float('-inf'))
        return F.softmax(s, dim=-1) @ v

class MultiHead(nn.Module):
    def __init__(self, ed, nh, bs):
        super().__init__()
        self.heads = nn.ModuleList([Head(ed, ed//nh, bs) for _ in range(nh)])
        self.proj  = nn.Linear(ed, ed)
    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.heads], dim=-1))

class MoEBlock(nn.Module):
    def __init__(self, ed, nh, bs, n_experts, expert_inner):
        super().__init__()
        self.ln1 = nn.LayerNorm(ed)
        self.attn = MultiHead(ed, nh, bs)
        self.ln2 = nn.LayerNorm(ed)
        self.moe = MoEMLP(ed, n_experts, expert_inner)

    def forward(self, x, hard=False):
        x = x + self.attn(self.ln1(x))
        moe_out, router_probs = self.moe(self.ln2(x), hard=hard)
        x = x + moe_out
        return x, router_probs

class TinyMoETransformer(nn.Module):
    def __init__(self, V, ed, nh, bs, L, n_experts, expert_inner):
        super().__init__()
        self.block_size = bs
        self.token_embed = nn.Embedding(V, ed)
        self.pos_embed   = nn.Embedding(bs, ed)
        self.blocks = nn.ModuleList([MoEBlock(ed, nh, bs, n_experts, expert_inner) for _ in range(L)])
        self.ln_final = nn.LayerNorm(ed)
        self.lm_head  = nn.Linear(ed, V)

    def forward(self, idx, targets=None, hard=False):
        B, T = idx.shape
        x = self.token_embed(idx) + self.pos_embed(torch.arange(T, device=idx.device))
        router_probs_all = []
        for block in self.blocks:
            x, rp = block(x, hard=hard)
            router_probs_all.append(rp)
        x = self.ln_final(x)
        logits = self.lm_head(x)
        if targets is None: return logits, None, router_probs_all
        ce = F.cross_entropy(logits.view(B*T, -1), targets.view(B*T))
        lb = sum(load_balance_loss(rp) for rp in router_probs_all) / len(router_probs_all)
        return logits, (ce, lb), router_probs_all

model = TinyMoETransformer(VOCAB_SIZE, EMBED_DIM, NUM_HEADS, BLOCK_SIZE, N_LAYERS, N_EXPERTS, EXPERT_INNER).to(device)
print(f'parameters: {sum(p.numel() for p in model.parameters()):,}')
for n, p in list(model.named_parameters())[:10]:
    print(f'  {n:50s} {tuple(p.shape)}')
print('  ...')


## Cell 5 — Training loop with load balance loss + best-val checkpoint

Standard cross-entropy plus the load balance auxiliary, weighted by `LB_LOSS_W=0.01`. The weight matters — too high and the router can't make decisions (always perfectly uniform); too low and the router collapses.

`0.01` is the Switch Transformer default and usually works. We'll monitor expert usage during training to verify the router is actually using all experts.


In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
history = []
best_val = float('inf')
best_state = None

@torch.no_grad()
def eval_model(n_batches=20):
    model.eval()
    out = {}
    for split in ('train', 'val'):
        ls = torch.zeros(n_batches)
        for k in range(n_batches):
            xb, yb = get_batch(split)
            _, (ce, _), _ = model(xb, yb, hard=False)
            ls[k] = ce.item()
        out[split] = ls.mean().item()
    # Also measure HARD-routing val loss (the actual deployment path)
    ls_hard = torch.zeros(n_batches)
    for k in range(n_batches):
        xb, yb = get_batch('val')
        _, (ce, _), _ = model(xb, yb, hard=True)
        ls_hard[k] = ce.item()
    out['val_hard'] = ls_hard.mean().item()
    # Expert usage (last batch only — illustrative)
    xb, yb = get_batch('val')
    _, _, rps = model(xb, yb, hard=False)
    usage = []
    for rp in rps:
        top1 = rp.argmax(dim=-1)
        usage.append([float((top1 == e).float().mean()) for e in range(N_EXPERTS)])
    out['usage'] = usage
    model.train()
    return out

for step in range(N_STEPS + 1):
    if step % EVAL_EVERY == 0:
        ev = eval_model()
        history.append((step, ev['train'], ev['val'], ev['val_hard'], ev['usage']))
        marker = ''
        if ev['val_hard'] < best_val:
            best_val = ev['val_hard']
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            marker = '  <-- new best (hard)'
        usage_str = ' | '.join(f'L{i}: ' + ','.join(f'{x:.2f}' for x in u) for i, u in enumerate(ev['usage']))
        print(f'step {step:>5} | train {ev["train"]:.4f} | val(soft) {ev["val"]:.4f} | val(hard) {ev["val_hard"]:.4f}{marker}')
        print(f'              expert usage [{usage_str}]')
        if step % (EVAL_EVERY*2) == 0: print()

    xb, yb = get_batch('train')
    _, (ce, lb), _ = model(xb, yb, hard=False)
    loss = ce + LB_LOSS_W * lb
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()

print(f'\nbest val (hard mode): {best_val:.4f}')
model.load_state_dict(best_state)


## Cell 6 — Sanity check: did the experts specialize?

The crucial question. If all experts are doing the same thing (router using them roughly uniformly but all experts producing similar outputs), MoE adds no value over a dense MLP. We want to see *different behavior per expert*.

### What we'll measure

For each expert: of the tokens routed to it, **what BPE tokens are most common?** If expert 0 mostly handles `the`, `and`, `of` and expert 1 mostly handles `lord`, `king`, `queen`, we have real specialization.


In [ ]:
@torch.no_grad()
def expert_token_histogram(model, layer_idx, n_batches=50):
    model.eval()
    expert_tokens = [Counter() for _ in range(N_EXPERTS)]
    for _ in range(n_batches):
        xb, _ = get_batch('val')
        # Forward through embeds + earlier blocks
        T = xb.shape[1]
        x = model.token_embed(xb) + model.pos_embed(torch.arange(T, device=device))
        for i, block in enumerate(model.blocks[:layer_idx]):
            x, _ = block(x, hard=True)
        # Now route through this layer's MoE
        x_ln = model.blocks[layer_idx].ln2(
            x + model.blocks[layer_idx].attn(model.blocks[layer_idx].ln1(x)))
        rp = F.softmax(model.blocks[layer_idx].moe.router(x_ln), dim=-1)
        top1 = rp.argmax(dim=-1)
        for e in range(N_EXPERTS):
            mask = (top1 == e)
            for tok_id in xb[mask].tolist():
                expert_tokens[e][tok_id] += 1
    model.train()
    return expert_tokens

print('=== Layer 0 expert specialization ===')
hist = expert_token_histogram(model, 0)
for e in range(N_EXPERTS):
    top5 = hist[e].most_common(5)
    tokens = ', '.join(f'{itos[tid]!r}' for tid, _ in top5)
    total = sum(hist[e].values())
    print(f'  Expert {e} ({total} tokens routed): top 5 = {tokens}')

print('\n=== Layer 1 expert specialization ===')
hist1 = expert_token_histogram(model, 1)
for e in range(N_EXPERTS):
    top5 = hist1[e].most_common(5)
    tokens = ', '.join(f'{itos[tid]!r}' for tid, _ in top5)
    total = sum(hist1[e].values())
    print(f'  Expert {e} ({total} tokens routed): top 5 = {tokens}')


## Cell 7 — Compare to nb07b's dense transformer

Side-by-side: nb07b's best val ≈ 3.04 nats/token. What does MoE achieve at the same EEPROM budget?

Both numbers measured *in hard inference mode* — since that's what the 6502 will run.


In [ ]:
steps, trains, vals_soft, vals_hard, _ = zip(*history)
plt.figure(figsize=(10, 5))
plt.plot(steps, trains, label='MoE train', alpha=0.7)
plt.plot(steps, vals_soft, label='MoE val (soft routing)', alpha=0.7)
plt.plot(steps, vals_hard, label='MoE val (HARD routing, deployment)', color='C2', linewidth=2)
plt.axhline(3.04, color='red', linestyle='--', alpha=0.7, label='nb07b dense transformer')
plt.xlabel('step'); plt.ylabel('cross-entropy loss (nats/token)')
plt.title('MoE-Tiny vs dense transformer (same EEPROM budget)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

print(f'\nFinal numbers:')
print(f'  MoE-Tiny best val (hard): {best_val:.4f}')
print(f'  nb07b transformer:        3.0400')
print(f'  delta: {best_val - 3.04:+.4f} nats  ({100*(best_val/3.04 - 1):+.2f}%)')
print('  (negative = MoE wins)')


## Cell 8 — Generate from the (hard top-1) MoE model

Crucial: generation uses **hard top-1 routing** (matches deployment). Quality should be similar to or better than nb07b. Watch for:

- Real words appearing (BPE doing its job).
- Less made-up token salad (theoretical MoE win).
- Smoother grammar within phrases (each expert specialized).


In [ ]:
@torch.no_grad()
def generate(model, prompt='', max_new_tokens=120, temperature=1.0):
    model.eval()
    ids = encode(prompt) if prompt else [1]
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -BLOCK_SIZE:]
        logits, _, _ = model(idx_cond, hard=True)
        probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
        nxt = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, nxt], dim=1)
    return decode(idx[0].tolist())

print('--- prompt="king", temp=1.0 ---')
print(generate(model, prompt='king', max_new_tokens=120))
print('\n--- prompt="romeo", temp=0.7 ---')
print(generate(model, prompt='romeo', max_new_tokens=120, temperature=0.7))


## Cell 9 — Quantize to int8 and re-check loss

Same per-tensor PTQ as nb07b. The router weights, expert weights, attention, and lm_head all get quantized. LayerNorm stays fp.


In [ ]:
def quantize_symmetric(t):
    max_abs = t.abs().max().item()
    if max_abs == 0: return torch.zeros_like(t, dtype=torch.int8), 1.0
    scale = max_abs / 127.0
    return torch.round(t / scale).clamp(-128, 127).to(torch.int8), scale

def dequantize(qt, s): return qt.to(torch.float32) * s

q_table = {}
for name, p in model.named_parameters():
    if 'ln' in name or 'norm' in name: continue
    if p.dim() < 1: continue
    qp, s = quantize_symmetric(p.data)
    q_table[name] = (qp, s)
    p.data.copy_(dequantize(qp, s))

# Re-eval in hard mode
model.eval()
with torch.no_grad():
    losses = torch.zeros(40)
    for k in range(40):
        xb, yb = get_batch('val')
        _, (ce, _), _ = model(xb, yb, hard=True)
        losses[k] = ce.item()
    val_int8 = losses.mean().item()

print(f'fp32 val (hard): {best_val:.4f}')
print(f'int8 val (hard): {val_int8:.4f}')
print(f'degradation:     {(val_int8 - best_val):+.4f} nats')


## Cell 10 — Pack `wozformer_moe.bin`

Similar layout to nb07b but with per-block expert tables. The 6502 firmware reads:

```
For each layer:
  1. LayerNorm input (fp16 gamma/beta from header).
  2. Self-attention (Q/K/V/proj weights).
  3. Add residual, LayerNorm.
  4. Compute router_logits = router_weights @ x   (small matmul, 4 outputs).
  5. Find argmax → expert_id.
  6. Run only experts[expert_id] (skip the other 3 entirely).
  7. Add residual.
```

Step 6 is where the cycle savings come from. The firmware doesn't even bother loading the unused experts from EEPROM.


In [ ]:
export_dir = Path('../export'); export_dir.mkdir(exist_ok=True)
out_path = export_dir / 'wozformer_moe.bin'

# Build tensor order
TENSOR_ORDER = ['token_embed.weight', 'pos_embed.weight']
for i in range(N_LAYERS):
    TENSOR_ORDER += [
        f'blocks.{i}.attn.heads.0.key.weight',
        f'blocks.{i}.attn.heads.0.query.weight',
        f'blocks.{i}.attn.heads.0.value.weight',
        f'blocks.{i}.attn.proj.weight', f'blocks.{i}.attn.proj.bias',
        f'blocks.{i}.moe.router.weight',
    ]
    for e in range(N_EXPERTS):
        TENSOR_ORDER += [
            f'blocks.{i}.moe.experts.{e}.0.weight',
            f'blocks.{i}.moe.experts.{e}.0.bias',
            f'blocks.{i}.moe.experts.{e}.2.weight',
            f'blocks.{i}.moe.experts.{e}.2.bias',
        ]
TENSOR_ORDER += ['lm_head.weight', 'lm_head.bias']

LN_TENSORS = []
for i in range(N_LAYERS):
    LN_TENSORS += [f'blocks.{i}.ln1.weight', f'blocks.{i}.ln1.bias',
                   f'blocks.{i}.ln2.weight', f'blocks.{i}.ln2.bias']
LN_TENSORS += ['ln_final.weight', 'ln_final.bias']

state = dict(model.state_dict())

buf = bytearray()
buf += b'WMOE'
buf += bytes([1, VOCAB_SIZE, EMBED_DIM, BLOCK_SIZE, NUM_HEADS, N_LAYERS, N_EXPERTS, EXPERT_INNER])
buf += struct.pack('<H', len(merges))

# Vocab + merges (so the Arduino can re-encode)
for tok in itos:
    b = tok.encode('utf-8')
    buf += bytes([len(b)]) + b
for (a, b), m in merges:
    for piece in (a, b, m):
        pb = piece.encode('utf-8')
        buf += bytes([len(pb)]) + pb

# int8 tensors
for name in TENSOR_ORDER:
    t = state[name]
    qt, s = quantize_symmetric(t)
    buf += struct.pack('<f', s)
    buf += qt.cpu().numpy().tobytes()

# fp32 LN params
for name in LN_TENSORS:
    t = state[name].cpu().numpy().astype(np.float32)
    buf += t.tobytes()

out_path.write_bytes(buf)

BUDGET = 32 * 1024
print(f'wrote {out_path}  ({len(buf):,} bytes)')
print(f'EEPROM budget (32 KB): {100*len(buf)/BUDGET:.1f}% used')
print(f'headroom: {BUDGET - len(buf):,} bytes')


## Cell 11 — Cycle budget per token (deployment, hard top-1)

What actually runs on the 6502 per generated token:


In [ ]:
T = BLOCK_SIZE; ed = EMBED_DIM; hs = ed; V = VOCAB_SIZE; L = N_LAYERS; ei = EXPERT_INNER; ne = N_EXPERTS

# Per-token (KV-cached: only the last position computed)
per_token = (
    L * (
        3 * 1 * ed * hs +           # Q,K,V projections (T_eff=1)
        1 * T * hs +                # QK^T (Q is 1, K is full T)
        1 * T * hs +                # softmax @ V
        1 * ed * ed +               # attn output projection
        1 * ed * ne +               # router
        1 * ed * ei +               # one expert: up projection
        1 * ei * ed                 # one expert: down projection
    )
    + 1 * ed * V                    # lm_head
)

print(f'per-token (KV cached, hard top-1): {per_token:,} mults')
print(f'  @80 cycles/mul on 6502: {per_token*80:,} cycles = {per_token*80/1e6*1000:.0f} ms @ 1 MHz')

print(f'\nCompare to nb07b: ~17,000 mults (~1,400 ms/token)')
print(f'MoE per-token is {per_token/17000:.2f}× nb07b\'s cost')


## Cell 12 — Save checkpoint


In [ ]:
ckpt = Path('../export/wozformer_moe.pt')
torch.save({
    'config': dict(vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, block_size=BLOCK_SIZE,
                   num_heads=NUM_HEADS, n_layers=N_LAYERS, n_experts=N_EXPERTS,
                   expert_inner=EXPERT_INNER),
    'model_state_int8_simulated': model.state_dict(),
    'quant_table': {k: (v[0].cpu().numpy().tolist(), v[1]) for k, v in q_table.items()},
    'itos': itos,
    'merges': [(list(p), m) for p, m in merges],
    'val_loss_int8_hard': val_int8,
    'val_loss_fp32_hard': best_val,
}, ckpt)
print(f'wrote {ckpt}  ({ckpt.stat().st_size:,} bytes)')


## Post-mortem — did sparse experts beat dense MLP?

### The honest answer depends on what you measured

| Metric | nb07b (dense) | nb09 (MoE) | Winner |
|---|---|---|---|
| Val loss (hard mode) | 3.04 | YOUR_VAL | ? |
| Per-token compute | ~17K mults | ~6K mults | **MoE (faster)** |
| EEPROM | ~29 KB | ~YOUR_KB | depends |
| Output quality | real words + filler | YOUR_OUTPUT | judge by eye |

### What the comparison tells you

- If **val(hard) < 3.04 AND expert specialization is real**: MoE is the better generative architecture. Ship `wozformer_moe.bin` in the generative slot.
- If **val(hard) > 3.04 OR experts all look identical in Cell 6**: MoE didn't pay off at this scale. Ship `wozformer_v2.bin` (transformer) instead.
- Either way: the **per-token cycle savings are real and unconditional** — MoE generation will be faster on the 6502 regardless of quality.

### What you learned

- The Switch Transformer pattern (top-1 routing + load balance loss) at the smallest scale anyone's tried it.
- Why MoE is a *capacity vs compute* tradeoff: more parameters, less compute per token.
- The specialization phenomenon — by examining which tokens go to which expert, you see the model auto-discover linguistic categories.

### Onwards

After you pick the winning generative architecture (transformer or MoE), the project is fully set for hardware:

- **Generative slot**: `wozformer_v2.bin` or `wozformer_moe.bin` (whichever won).
- **Retrieval slot**: `wozformer_rag.bin` (from nb08).

Both flashed to EEPROMs. Mode switch on the breadboard toggles between them. C reference + Arduino programmer + 6502 firmware next.
